# Anima: grouped conditionings (three attributes asked for at once: do they compose, and do they pull on each other?)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AbstractEyes/anima-trainer/blob/main/notebooks/anima_colab_groups.ipynb)

The counterpoint pairs found booru tags that each move one attribute of an
**[Anima](https://huggingface.co/circlestone-labs/Anima)** picture, and the sliders turned them into dials. A description
usually asks for several attributes together, so this notebook asks for three at once and checks three things: each one
still moves its own attribute, it keeps most of what it does alone, and no attribute pulls on another.

| pair | + tag | - tag |
|---|---|---|
| hair colour | blonde hair | black hair |
| expression | smile | frown |
| eye colour | red eyes | blue eyes |

**Two ways of asking.** *Words*: the three tags written in one prompt. *Sliders*: the three single-tag predictions
composed at every sampling step, v = v(plain) + sum over the tags of (v(tag) - v(plain)), each at amount 1 (one tag
alone is the earlier slider). Every side of every pair is drawn: 8 groups x 2 ways x 16 cells (eight adult characters x
two seeds).

**Nothing is drawn twice.** The plain prompt's pictures and each tag's own pictures on the same characters come from
[AbstractPhil/geolip-beatrix-anima-data](https://huggingface.co/datasets/AbstractPhil/geolip-beatrix-anima-data) (the
updated counterpoint sets); the 16 new group sets are saved there under `sets/cgr2_*` as soon as each is drawn.

**Content screen.** The tagger rates every picture and reads its age tags: a picture rated questionable or explicit, or
read as child, loli or aged down above 0.2, is left off the sheets and its set is not uploaded. The fourth character's
pictures are drawn with "child, loli, aged down" added to the negative prompt.

**The rule (fixed before the run).** *Survival*: in each group, each pair's score moves that group's way against the plain
prompt in at least 12 of the 16 cells, beyond 3 standard errors; a pair survives the group when it does so in all 8.
*Kept*: the group's move over the single tag's move, at least 0.75. *Overlap*: a pair's score shifts by more than a
third of its range when another pair flips side.

**Cost.** 256 new pictures at 768 px and 30 steps: the words at a plain picture's cost, the sliders about 2.5 times it
(five predictions a step instead of two). About 25 minutes on the RTX PRO 6000. Setup: the dependencies, the three
model files (~5.6 GB) and the tagger (1.3 GB). Results: `experiments/e053_anima_grouped_conditionings` of
[AbstractPhil/geolip-beatrix-anima](https://huggingface.co/AbstractPhil/geolip-beatrix-anima).


## 0 · Confirm the GPU and disk

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!df -h /content | tail -1

## 1 · Bootstrap: the runner at its release, and the dependencies (restart ONCE the first time)

Clones the runner at the release this notebook was written for, and the diffusion-pipe fork beside it (its Anima model
code draws the pictures), then installs the dependencies. **Run the cell; if the kernel restarts, run it again** (it
then skips the install).

In [ ]:
import os, sys, subprocess
TAG = "v0.5.0"                       # this notebook's release of the runner
REPO = os.environ.get("ANIMA_REPO", "/content/anima-trainer")
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "--quiet", "https://github.com/AbstractEyes/anima-trainer.git", REPO], check=True)
subprocess.run(["git", "-C", REPO, "fetch", "--quiet", "--tags", "origin"], check=True)
subprocess.run(["git", "-C", REPO, "checkout", "--quiet", TAG], check=True)
sys.path.insert(0, REPO)
import anima_colab
if anima_colab.install(REPO, dp_url=anima_colab.DP_FORK_URL, similarity=False):   # + the diffusion-pipe fork
    os.kill(os.getpid(), 9)             # -> restart; reconnect and RE-RUN this cell (then it skips)

## 2 · Set up

Checks the GPU, finds the fork and downloads the three model files. Needs a **write** `HF_TOKEN` in Colab Secrets: the
results and the drawn sets are uploaded.

In [ ]:
from geolip_anima_trainer.anima_runner import AnimaRunner
a = AnimaRunner()      # Anima-Base v1.0 at 768 px; results -> AbstractPhil/geolip-beatrix-anima, sets -> ...-anima-data
a.setup()              # env + HF login + GPU check + the fork + the three model files (~5.6 GB)

## 3 · The grouped conditionings

Prints the job's size first (which sets come from the data repo, how many pictures it draws), then time spent and time
left with one line per set, and at the end one line per way of asking (words, sliders): the verdict, how many of the 8
groups each pair survives, the share of each single tag's move kept, and any overlap. A run already done is skipped
(`force=True` reruns the reads; the sets come back from the data repo).


In [ ]:
meta = a.run_counterpoint_groups()     # hair colour, expression, eye colour together: 8 groups x (words, sliders)


## Notes

- **Where things land:** `experiments/e053_anima_grouped_conditionings/` in the experiments repo (README with the
  question, the design, the rule fixed before the run and the result; `meta.json`; `result.json` with every cell's score
  on every pair in every set; `tagger_probs.npz`; `sheet_words.jpg` and `sheet_sliders.jpg`). The sets:
  `sets/cgr2_<tags>_<words|sliders>-<key>/` in the data repo. Locally: `/content/anima_data/`.
- **After a disconnect:** run steps 1-3 again. Finished sets come back from the data repo; only the unfinished set is
  drawn again.
- **The pictures** are outputs of Anima-Base v1.0 (CircleStone Labs Non-Commercial License: outputs are not derivatives
  of the model; the restriction applies to the model and its derivatives, such as LoRAs).
